# Parkinson's UPDRS Prediction — Model Building

**Task:** Multi-output Regression — predict `motor_UPDRS` and `total_UPDRS` simultaneously.

**Pipeline:**
1. Load preprocessed data
2. Baseline: Ridge / Lasso
3. SVR (RBF kernel)
4. Random Forest
5. Gradient Boosting (XGBoost)
6. MLP (Neural Network)
7. Model comparison + Feature Importance
8. Residual analysis & inference on test set

## 0. Setup

In [ ]:
# !pip install -q ucimlrepo xgboost

In [ ]:
import warnings
import pickle
import os

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import randint, uniform, pearsonr

from sklearn.linear_model import Ridge, RidgeCV, Lasso, LassoCV
from sklearn.svm import SVR
from sklearn.ensemble import RandomForestRegressor
from sklearn.multioutput import MultiOutputRegressor
from sklearn.neural_network import MLPRegressor
from sklearn.model_selection import GroupKFold, GroupShuffleSplit, RandomizedSearchCV
from sklearn.metrics import (
    mean_squared_error,
    mean_absolute_error,
    mean_absolute_percentage_error,
    r2_score,
)
from sklearn.preprocessing import StandardScaler
import xgboost as xgb

warnings.filterwarnings("ignore")
pd.set_option("display.float_format", lambda x: f"{x:.4f}")
plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "axes.grid": True,
    "grid.alpha": 0.3,
    "font.size": 11,
})

PALETTE = sns.color_palette("muted")
SEED = 42
print("Imports OK")

## 1. Load Preprocessed Data

In [ ]:
data = pd.read_csv("preprocessed_data.csv")
data = data.rename(columns={
    "Unnamed: 0": "subject#",
    "Jitter(avg)": "Jitter_avg",
    "Shimmer(avg)": "Shimmer_avg",
})

TARGETS = ["motor_UPDRS", "total_UPDRS"]
features = [c for c in data.columns if c not in TARGETS + ["subject#"]]

X = data[features]
y = data[TARGETS]

# Group-aware split to prevent data leakage across subjects
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
train_idx, test_idx = next(gss.split(X, y, groups=data["subject#"]))

X_train_raw, X_test_raw = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

scaler = StandardScaler()
X_train = pd.DataFrame(scaler.fit_transform(X_train_raw), columns=features)
X_test  = pd.DataFrame(scaler.transform(X_test_raw),  columns=features)

train_subjects = data.iloc[train_idx]["subject#"].values

# Numpy arrays for sklearn helpers
X_tr_arr = X_train.values
y_tr_arr = y_train.values

print(f"Train: {X_train.shape} | Test: {X_test.shape}")
print(f"Features: {features}")

## 2. Helper Functions

In [ ]:
def evaluate(y_true, y_pred, split="Test"):
    """Compute RMSE, MAE, R², MAPE and Pearson-r for both targets."""
    results = {}
    for i, col in enumerate(TARGETS):
        yt = y_true.iloc[:, i] if hasattr(y_true, "iloc") else y_true[:, i]
        yp = y_pred[:, i] if y_pred.ndim == 2 else y_pred
        results[col] = {
            "RMSE":      np.sqrt(mean_squared_error(yt, yp)),
            "MAE":       mean_absolute_error(yt, yp),
            "R²":        r2_score(yt, yp),
            "MAPE":      mean_absolute_percentage_error(yt, yp),
            "Pearson_r": pearsonr(yt, yp)[0],
        }
    df_res = pd.DataFrame(results).T
    print(f"\n── {split} Set ──")
    print(df_res.round(4))
    return df_res


def cv_score(model, X, y, groups, n_splits=5, label=""):
    """GroupKFold cross-validation — returns mean RMSE and R² per target."""
    gkf = GroupKFold(n_splits=n_splits)
    rmse_motor, rmse_total, r2_motor, r2_total = [], [], [], []

    for tr, val in gkf.split(X, y, groups):
        model.fit(X[tr], y[tr])
        pred = model.predict(X[val])
        rmse_motor.append(np.sqrt(mean_squared_error(y[val, 0], pred[:, 0])))
        rmse_total.append(np.sqrt(mean_squared_error(y[val, 1], pred[:, 1])))
        r2_motor.append(r2_score(y[val, 0], pred[:, 0]))
        r2_total.append(r2_score(y[val, 1], pred[:, 1]))

    print(f"\n{'='*50}")
    print(f"[{label}] GroupKFold CV ({n_splits} folds)")
    print(f"  motor_UPDRS — RMSE: {np.mean(rmse_motor):.4f} ± {np.std(rmse_motor):.4f}  |  R²: {np.mean(r2_motor):.4f}")
    print(f"  total_UPDRS — RMSE: {np.mean(rmse_total):.4f} ± {np.std(rmse_total):.4f}  |  R²: {np.mean(r2_total):.4f}")

    return {
        "motor_RMSE": np.mean(rmse_motor), "total_RMSE": np.mean(rmse_total),
        "motor_R2":   np.mean(r2_motor),   "total_R2":   np.mean(r2_total),
    }


# Accumulators
cv_results   = {}
test_results = {}
print("Helpers ready.")

## 3. Baseline: Ridge & Lasso

Ridge uses L2 regularization — well-suited when multicollinearity is present.  
Lasso uses L1 — can zero out features, useful for implicit feature selection.

In [ ]:
# Ridge — baseline (alpha=1) and auto-tuned via RidgeCV
ridge_base = MultiOutputRegressor(Ridge(alpha=1.0), n_jobs=-1)
cv_results["Ridge_Baseline"] = cv_score(ridge_base, X_tr_arr, y_tr_arr, train_subjects, label="Ridge Baseline")
ridge_base.fit(X_tr_arr, y_tr_arr)
test_results["Ridge_Baseline"] = evaluate(y_test, ridge_base.predict(X_test.values), "Ridge Baseline")

ridge_cv = MultiOutputRegressor(RidgeCV(alphas=np.logspace(-3, 3, 10)), n_jobs=-1)
cv_results["Ridge_Tuned"] = cv_score(ridge_cv, X_tr_arr, y_tr_arr, train_subjects, label="Ridge Tuned")
ridge_cv.fit(X_tr_arr, y_tr_arr)
test_results["Ridge_Tuned"] = evaluate(y_test, ridge_cv.predict(X_test.values), "Ridge Tuned")

In [ ]:
# Lasso — find best alpha per target, then fit a shared model
lasso_cv_motor = LassoCV(cv=5, random_state=SEED).fit(X_tr_arr, y_tr_arr[:, 0])
lasso_cv_total = LassoCV(cv=5, random_state=SEED).fit(X_tr_arr, y_tr_arr[:, 1])
best_alpha = max(lasso_cv_motor.alpha_, lasso_cv_total.alpha_)
print(f"Best alpha — motor: {lasso_cv_motor.alpha_:.4f}  |  total: {lasso_cv_total.alpha_:.4f}  |  chosen: {best_alpha:.4f}")

lasso = MultiOutputRegressor(Lasso(alpha=best_alpha), n_jobs=-1)
cv_results["Lasso"] = cv_score(lasso, X_tr_arr, y_tr_arr, train_subjects, label="Lasso")
lasso.fit(X_tr_arr, y_tr_arr)
test_results["Lasso"] = evaluate(y_test, lasso.predict(X_test.values), "Lasso")

# Coefficient inspection
coef_df = pd.DataFrame({
    "feature":     X_train.columns,
    "coef_motor":  lasso.estimators_[0].coef_,
    "coef_total":  lasso.estimators_[1].coef_,
})
print("\n── Lasso coefficients (sorted by |motor coef|) ──")
print(coef_df.sort_values("coef_motor", key=abs, ascending=False).to_string(index=False))

## 4. SVR (RBF Kernel)

In [ ]:
# Baseline SVR
svr_base = MultiOutputRegressor(SVR(kernel="rbf", C=1.0), n_jobs=-1)
cv_results["SVR_Baseline"] = cv_score(svr_base, X_tr_arr, y_tr_arr, train_subjects, label="SVR Baseline")
svr_base.fit(X_tr_arr, y_tr_arr)
test_results["SVR_Baseline"] = evaluate(y_test, svr_base.predict(X_test.values), "SVR Baseline")

# Tuned SVR via RandomizedSearch
gkf = GroupKFold(n_splits=5)
param_dist_svr = {
    "estimator__C":       uniform(0.1, 20),
    "estimator__epsilon": uniform(0.01, 0.5),
    "estimator__gamma":   ["scale", "auto"],
}
svr_search = RandomizedSearchCV(
    MultiOutputRegressor(SVR(kernel="rbf"), n_jobs=-1),
    param_dist_svr, n_iter=10,
    cv=gkf.split(X_tr_arr, y_tr_arr, train_subjects),
    scoring="neg_root_mean_squared_error",
    n_jobs=-1, random_state=SEED,
)
svr_search.fit(X_tr_arr, y_tr_arr)

cv_results["SVR_Tuned"]   = cv_score(svr_search.best_estimator_, X_tr_arr, y_tr_arr, train_subjects, label="SVR Tuned")
test_results["SVR_Tuned"] = evaluate(y_test, svr_search.best_estimator_.predict(X_test.values), "SVR Tuned")

## 5. Random Forest

In [ ]:
# Baseline Random Forest
rf_base = MultiOutputRegressor(
    RandomForestRegressor(n_estimators=200, max_features="sqrt",
                          min_samples_leaf=5, random_state=SEED, n_jobs=-1),
    n_jobs=1,
)
cv_results["RandomForest"] = cv_score(rf_base, X_tr_arr, y_tr_arr, train_subjects, label="RandomForest")
rf_base.fit(X_tr_arr, y_tr_arr)
test_results["RandomForest"] = evaluate(y_test, rf_base.predict(X_test.values), "RandomForest")

# Tuned via RandomizedSearch
param_dist_rf = {
    "estimator__n_estimators":    randint(100, 400),
    "estimator__max_depth":       [None, 10, 20, 30],
    "estimator__min_samples_leaf": randint(2, 15),
    "estimator__max_features":    ["sqrt", "log2", 0.5],
}
rf_search = RandomizedSearchCV(
    MultiOutputRegressor(RandomForestRegressor(random_state=SEED, n_jobs=-1), n_jobs=1),
    param_dist_rf, n_iter=20,
    cv=gkf.split(X_tr_arr, y_tr_arr, train_subjects),
    scoring="neg_root_mean_squared_error",
    n_jobs=-1, random_state=SEED, verbose=0,
)
rf_search.fit(X_tr_arr, y_tr_arr)
print("Best RF params:", rf_search.best_params_)
print(f"Best CV RMSE  : {-rf_search.best_score_:.4f}")

rf_best = rf_search.best_estimator_
test_results["RandomForest_Tuned"] = evaluate(y_test, rf_best.predict(X_test.values), "RandomForest (Tuned)")

## 6. Gradient Boosting (XGBoost)

In [ ]:
# Baseline XGBoost
xgb_base = MultiOutputRegressor(xgb.XGBRegressor(random_state=SEED), n_jobs=-1)
cv_results["XGB_Baseline"] = cv_score(xgb_base, X_tr_arr, y_tr_arr, train_subjects, label="XGB Baseline")
xgb_base.fit(X_tr_arr, y_tr_arr)
test_results["XGB_Baseline"] = evaluate(y_test, xgb_base.predict(X_test.values), "XGB Baseline")

# Tuned via RandomizedSearch
param_dist_xgb = {
    "estimator__n_estimators":   randint(100, 500),
    "estimator__learning_rate":  uniform(0.01, 0.2),
    "estimator__max_depth":      randint(3, 7),
    "estimator__subsample":      uniform(0.6, 0.4),
}
xgb_search = RandomizedSearchCV(
    MultiOutputRegressor(xgb.XGBRegressor(random_state=SEED), n_jobs=-1),
    param_dist_xgb, n_iter=10,
    cv=gkf.split(X_tr_arr, y_tr_arr, train_subjects),
    scoring="neg_root_mean_squared_error",
    n_jobs=-1, random_state=SEED,
)
xgb_search.fit(X_tr_arr, y_tr_arr)

cv_results["XGB_Tuned"]   = cv_score(xgb_search.best_estimator_, X_tr_arr, y_tr_arr, train_subjects, label="XGB Tuned")
test_results["XGB_Tuned"] = evaluate(y_test, xgb_search.best_estimator_.predict(X_test.values), "XGB Tuned")

## 7. MLP (Neural Network)

In [ ]:
param_dist_mlp = {
    "estimator__hidden_layer_sizes": [(64, 32), (128, 64, 32), (100,)],
    "estimator__alpha":              uniform(0.0001, 0.05),
    "estimator__learning_rate_init": [0.001, 0.0005],
}
mlp_search = RandomizedSearchCV(
    MultiOutputRegressor(MLPRegressor(max_iter=500, early_stopping=True, random_state=SEED)),
    param_dist_mlp, n_iter=5,
    cv=gkf.split(X_tr_arr, y_tr_arr, train_subjects),
    scoring="neg_root_mean_squared_error",
    n_jobs=-1, random_state=SEED,
)
mlp_search.fit(X_tr_arr, y_tr_arr)
print("Best MLP params:", mlp_search.best_params_)

cv_results["MLP_Tuned"]   = cv_score(mlp_search.best_estimator_, X_tr_arr, y_tr_arr, train_subjects, label="MLP Tuned")
test_results["MLP_Tuned"] = evaluate(y_test, mlp_search.best_estimator_.predict(X_test.values), "MLP Tuned")

## 8. Model Comparison

In [ ]:
# CV summary table
cv_df = pd.DataFrame(cv_results).T
cv_df.index.name = "Model"
print("=== GroupKFold CV Results (mean across 5 folds) ===")
print(cv_df[["motor_RMSE", "total_RMSE", "motor_R2", "total_R2"]].round(4))

# Test set summary table
rows = [
    {
        "Model":      name,
        "motor_RMSE": df.loc["motor_UPDRS", "RMSE"],
        "total_RMSE": df.loc["total_UPDRS",  "RMSE"],
        "motor_R²":   df.loc["motor_UPDRS", "R²"],
        "total_R²":   df.loc["total_UPDRS",  "R²"],
    }
    for name, df in test_results.items()
]
test_df = pd.DataFrame(rows).set_index("Model")
print("\n=== Test Set Results ===")
print(test_df.round(4))

best_model = test_df["motor_RMSE"].idxmin()
print(f"\n✓ Best model (lowest motor_RMSE): {best_model}")

In [ ]:
# Bar chart: CV RMSE and R² for all models
fig, axes = plt.subplots(1, 2, figsize=(14, 5))
models_cv = list(cv_results.keys())
x, w = np.arange(len(models_cv)), 0.35

for ax, (m_key, t_key), ylabel in zip(
    axes,
    [("motor_RMSE", "total_RMSE"), ("motor_R2", "total_R2")],
    ["RMSE (lower is better)", "R² (higher is better)"],
):
    motor_vals = [cv_results[m][m_key] for m in models_cv]
    total_vals = [cv_results[m][t_key] for m in models_cv]
    b1 = ax.bar(x - w / 2, motor_vals, w, label="motor_UPDRS", color=PALETTE[0], alpha=0.85)
    b2 = ax.bar(x + w / 2, total_vals, w, label="total_UPDRS", color=PALETTE[1], alpha=0.85)
    ax.set_xticks(x)
    ax.set_xticklabels(models_cv, rotation=25, ha="right")
    ax.set_title(f"CV {ylabel}", fontweight="bold")
    ax.legend()
    for bar in list(b1) + list(b2):
        ax.text(
            bar.get_x() + bar.get_width() / 2,
            bar.get_height() + 0.002,
            f"{bar.get_height():.3f}",
            ha="center", va="bottom", fontsize=8,
        )

plt.suptitle("Cross-validation Performance Comparison", fontsize=13, fontweight="bold")
plt.tight_layout()
plt.show()

## 9. Feature Importance

In [ ]:
feat_names = list(X_train.columns)

xgb_best = xgb_search.best_estimator_
fi_xgb_motor = pd.Series(xgb_best.estimators_[0].feature_importances_, index=feat_names).sort_values()
fi_xgb_total = pd.Series(xgb_best.estimators_[1].feature_importances_, index=feat_names).sort_values()
fi_rf_motor  = pd.Series(rf_best.estimators_[0].feature_importances_,  index=feat_names).sort_values()
fi_rf_total  = pd.Series(rf_best.estimators_[1].feature_importances_,  index=feat_names).sort_values()

fig, axes = plt.subplots(2, 2, figsize=(14, 10))
fig.suptitle("Feature Importance Comparison", fontsize=14, fontweight="bold")

titles = [
    "XGBoost Tuned — motor_UPDRS", "XGBoost Tuned — total_UPDRS",
    "RandomForest Tuned — motor_UPDRS", "RandomForest Tuned — total_UPDRS",
]
for ax, fi, title in zip(axes.flatten(), [fi_xgb_motor, fi_xgb_total, fi_rf_motor, fi_rf_total], titles):
    colors = [PALETTE[0] if v >= fi.median() else PALETTE[2] for v in fi.values]
    ax.barh(fi.index, fi.values, color=colors)
    ax.set_title(title, fontweight="bold", fontsize=11)
    ax.set_xlabel("Importance")
    for i, v in enumerate(fi.values):
        ax.text(v + 0.001, i, f"{v:.3f}", va="center", fontsize=8)

plt.tight_layout()
plt.show()

## 10. Residual Analysis — Best Model

In [ ]:
pred_best  = xgb_search.best_estimator_.predict(X_test.values)
y_test_arr = y_test.values

fig, axes = plt.subplots(2, 3, figsize=(16, 9))
fig.suptitle(f"Residual Analysis — Best Model: {best_model}", fontsize=13, fontweight="bold")

for i, col in enumerate(TARGETS):
    residuals = y_test_arr[:, i] - pred_best[:, i]

    # Actual vs Predicted
    ax = axes[i, 0]
    ax.scatter(y_test_arr[:, i], pred_best[:, i], alpha=0.4, s=15, color=PALETTE[i])
    lims = [min(y_test_arr[:, i].min(), pred_best[:, i].min()),
            max(y_test_arr[:, i].max(), pred_best[:, i].max())]
    ax.plot(lims, lims, "r--", linewidth=1.5, label="Perfect fit")
    ax.set_xlabel(f"Actual {col}"); ax.set_ylabel("Predicted")
    ax.set_title(f"Actual vs Predicted: {col}")
    ax.legend()

    # Residuals vs Predicted
    ax = axes[i, 1]
    ax.scatter(pred_best[:, i], residuals, alpha=0.4, s=15, color=PALETTE[i])
    ax.axhline(0, color="red", linestyle="--", linewidth=1.5)
    ax.set_xlabel("Predicted"); ax.set_ylabel("Residual")
    ax.set_title(f"Residuals vs Predicted: {col}")

    # Residual distribution
    ax = axes[i, 2]
    sns.histplot(residuals, kde=True, ax=ax, color=PALETTE[i])
    ax.axvline(0, color="red", linestyle="--")
    ax.set_title(f"Residual Distribution: {col}")
    ax.set_xlabel("Residual")

plt.tight_layout()
plt.show()

## 11. Save Best Model

In [ ]:
os.makedirs("models", exist_ok=True)

with open("models/scaler.pkl", "wb") as f:
    pickle.dump(scaler, f)
print("✓ Saved models/scaler.pkl")

with open("models/best_model_xgb.pkl", "wb") as f:
    pickle.dump(xgb_search.best_estimator_, f)
print(f"✓ Saved models/best_model_xgb.pkl  ({best_model})")

print("=" * 60)
print(f"FINAL TEST SET PERFORMANCE — {best_model}")
print("=" * 60)
evaluate(y_test, pred_best, "Final")

## 12. Inference on New Samples

Use the best XGBoost model to predict on 20 held-out test samples.

In [ ]:
def run_inference_samples(model, X_test_df, y_test_df, n_samples=20):
    """Run inference on the first n_samples rows and return a comparison DataFrame."""
    X_sample = X_test_df.head(n_samples)
    y_sample = y_test_df.head(n_samples)
    preds = model.predict(X_sample.values)

    df = pd.DataFrame({
        "Actual Motor": y_sample["motor_UPDRS"].values,
        "Pred Motor":   preds[:, 0],
        "Actual Total": y_sample["total_UPDRS"].values,
        "Pred Total":   preds[:, 1],
    })
    df["Error Motor"] = (df["Actual Motor"] - df["Pred Motor"]).abs()
    df["Error Total"] = (df["Actual Total"] - df["Pred Total"]).abs()
    return df


inference_results = run_inference_samples(xgb_search.best_estimator_, X_test, y_test, n_samples=20)
print("=== Inference — first 20 test samples ===")
display(inference_results.round(3))

## 13. Ablation: RobustScaler + Top-6 Features

Re-train with only the six most important features identified by XGBoost and swap  
`StandardScaler` for `RobustScaler` to reduce the influence of outliers.  
Compare error counts (|error| > 10) before and after.

In [ ]:
TOP_FEATURES = ["age", "DFA", "HNR", "Shimmer_avg", "test_time", "RPDE"]

X_abl      = data[TOP_FEATURES]
gss_abl    = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=SEED)
tr_idx_abl, te_idx_abl = next(gss_abl.split(X_abl, y, groups=data["subject#"]))

X_tr_abl_raw, X_te_abl_raw = X_abl.iloc[tr_idx_abl], X_abl.iloc[te_idx_abl]
y_tr_abl,     y_te_abl     = y.iloc[tr_idx_abl],     y.iloc[te_idx_abl]

rob_scaler  = RobustScaler()  # imported at top; add if missing: from sklearn.preprocessing import RobustScaler
X_tr_abl    = pd.DataFrame(rob_scaler.fit_transform(X_tr_abl_raw), columns=TOP_FEATURES)
X_te_abl    = pd.DataFrame(rob_scaler.transform(X_te_abl_raw),     columns=TOP_FEATURES)

# Strip 'estimator__' prefix from tuned params for direct use
clean_params = {k.replace("estimator__", ""): v for k, v in xgb_search.best_params_.items()}

xgb_ablation = MultiOutputRegressor(xgb.XGBRegressor(**clean_params, random_state=SEED))
xgb_ablation.fit(X_tr_abl, y_tr_abl)
pred_ablation = xgb_ablation.predict(X_te_abl.values)

In [ ]:
print("── Original model (StandardScaler, all features) ──")
evaluate(y_test, pred_best, split="Original Test")

print("\n── Ablation model (RobustScaler, top-6 features) ──")
evaluate(y_te_abl, pred_ablation, split="Ablation Test")

errors_before = (np.abs(y_test.values  - pred_best)     > 10).sum()
errors_after  = (np.abs(y_te_abl.values - pred_ablation) > 10).sum()
print(f"\nPredictions with |error| > 10:  before = {errors_before}  |  after = {errors_after}")

## 14. Save Final Model

In [ ]:
os.makedirs("models", exist_ok=True)

with open("models/scaler_robust.pkl", "wb") as f:
    pickle.dump(rob_scaler, f)
print("✓ Saved models/scaler_robust.pkl")

with open("models/final_model_xgb.pkl", "wb") as f:
    pickle.dump(xgb_ablation, f)
print(f"✓ Saved models/final_model_xgb.pkl  ({best_model})")

print("=" * 60)
print(f"FINAL TEST SET PERFORMANCE — {best_model}")
print("=" * 60)
evaluate(y_te_abl, pred_ablation, "Final")